# 数据探索

本教程演示缺失值统计、异常值识别、描述性统计和相关性分析。代码使用项目根目录下的 `data/`，因此 Notebook 从不同目录启动时也能定位数据。


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print


RANDOM_STATE = 42


def find_project_root(start=None):
    """Find the project root by walking upward to the data directory.

    Args:
        start: Optional start path. Defaults to the current working directory.

    Returns:
        Project root path containing the `data` directory.

    Raises:
        FileNotFoundError: If no parent directory contains `data`.
    """
    current = Path.cwd() if start is None else Path(start).resolve()
    for path in [current, *current.parents]:
        if (path / "data").exists():
            return path
    raise FileNotFoundError("未找到包含 data/ 的项目根目录")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


## 1. 缺失值统计


In [ ]:
sample = pd.DataFrame(
    {
        "A": [1.0, np.nan, 3.0, np.nan],
        "B": [np.nan, np.nan, np.nan, np.nan],
        "C": ["x", "y", None, None],
        "D": [np.nan, np.nan, np.nan, np.nan],
    }
)

missing_summary = pd.DataFrame(
    {
        "missing_count": sample.isna().sum(),
        "missing_rate": sample.isna().mean(),
    }
)

all_missing_columns = sample.columns[sample.isna().all(axis=0)].tolist()
all_missing_rows = sample.index[sample.isna().all(axis=1)].tolist()

display(sample)
display(missing_summary)
print(f"全为空值的列数: {len(all_missing_columns)}，列名: {all_missing_columns}")
print(f"全为空值的行数: {len(all_missing_rows)}，行号: {all_missing_rows}")


## 2. 箱线图识别异常值


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
values = rng.normal(loc=0, scale=1, size=1000)

q1, q3 = np.quantile(values, [0.25, 0.75])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = values[(values < lower_bound) | (values > upper_bound)]

fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
ax.boxplot(values, vert=True, patch_artist=True, showmeans=True)
ax.scatter(np.ones(len(outliers)), outliers, color="crimson", s=18, label="异常点")
ax.set_title("正态分布样本箱线图")
ax.set_ylabel("取值")
ax.legend()
plt.show()

print(f"异常值数量: {len(outliers)}")
print(f"异常值边界: [{lower_bound:.3f}, {upper_bound:.3f}]")


## 3. 成绩数据描述性统计


In [ ]:
scores = pd.read_csv(DATA_DIR / "scores.csv")
numeric_scores = scores.select_dtypes(include="number")

summary = numeric_scores.describe().T
summary["range"] = numeric_scores.max() - numeric_scores.min()
summary["coefficient_of_variation"] = numeric_scores.std() / numeric_scores.mean()
summary["iqr"] = numeric_scores.quantile(0.75) - numeric_scores.quantile(0.25)

display(scores.head())
display(summary.round(3))


## 4. 相关性分析


In [ ]:
turnover = pd.read_csv(DATA_DIR / "turnover.csv")
correlation = turnover.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
turnover.plot.scatter(x="年广告费投入", y="月均销售额", ax=ax)
ax.set_title("广告费投入与月均销售额")
ax.grid(alpha=0.25)
plt.show()

display(turnover)
display(correlation.round(3))
